# Structuring Traces for Annotation Queues

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/comet-ml/opik-examples/blob/main/guides/annotation_queues_with_context/annotation_queues_with_context.ipynb)

Opik's [annotation queues](https://www.comet.com/docs/opik/evaluation/advanced/annotation_queues) route traces to subject-matter experts (SMEs) for human review. Whether a reviewer can judge an answer quickly depends on how the trace is structured — and a trace gives you four distinct places to put data:

| Field | What to put there |
|---|---|
| `input` | The user's question or request |
| `output` | The final answer only — not the full pipeline state |
| `metadata` | Supporting context a reviewer needs to verify quality (retrieved docs, sources) |
| child `spans` | Every sub-step (`@opik.track`), preserving full technical detail |

A common default is to return the whole pipeline dict — answer, retrieved documents, the built prompt — as the trace `output`. That buries the answer a reviewer needs to score under pipeline internals. Keeping `output` to the user-facing answer and moving supporting context into `metadata` (with full detail in spans) keeps the review surface focused while losing nothing.

**What you'll learn:**

- How to structure a traced RAG pipeline so input and output stay clean for review
- How to surface retrieved context in metadata using `opik_context.update_current_trace()`
- How to create annotation queues programmatically and populate them with traces
- The post-hoc enrichment pattern for existing traces

In [ ]:
%pip install --quiet --upgrade opik

In [ ]:
import opik

OPIK_PROJECT_NAME = "annotation-queues"

# Reads OPIK_API_KEY and OPIK_WORKSPACE from the environment (or prompts); targets Opik Cloud
opik.configure(project_name=OPIK_PROJECT_NAME)

## 1. Building the traced RAG pipeline

The key principle: **separate the user-facing answer from internal pipeline state.**

- `output` — the final answer only, not the full pipeline dict
- `metadata` — retrieval context for reviewers, set with `opik_context.update_current_trace()`
- child spans — every sub-step decorated with `@opik.track`; full detail preserved without crowding the output

Tag each sub-step with `type=` on `@opik.track` — `"tool"` for retrieval, `"llm"` for the model call — so its span renders with the right treatment in the Opik UI.

The example uses a mock retriever and a mock LLM so no API keys are required beyond Opik. Replace `generate()` with your actual LLM call (e.g. `openai.chat.completions.create`).

In [ ]:
MOCK_CORPUS = {
    "What is our leave policy?": [
        {"text": "Employees are entitled to 25 days of annual leave per year, plus public holidays.", "source": "HR Handbook v2.4, §3.1"},
        {"text": "Leave requests must be submitted at least 2 weeks in advance via the HR portal.", "source": "HR Handbook v2.4, §3.2"},
    ],
    "How do I request flexible working?": [
        {"text": "Flexible working requests should be submitted to your line manager using form FW-01.", "source": "Flexible Working Policy, §2"},
        {"text": "Requests will be reviewed within 28 days and approved or declined in writing.", "source": "Flexible Working Policy, §4"},
    ],
    "What are the performance review criteria?": [
        {"text": "Performance is assessed across four dimensions: delivery, collaboration, growth, and values.", "source": "Performance Framework 2024, §1"},
        {"text": "Annual reviews take place in Q1. Mid-year check-ins are mandatory for all staff.", "source": "Performance Framework 2024, §3"},
    ],
}

QUESTIONS = list(MOCK_CORPUS.keys())


@opik.track(type="tool")
def retrieve(question: str) -> list[dict]:
    return MOCK_CORPUS.get(question, [{"text": "No relevant documents found.", "source": "N/A"}])


@opik.track(type="llm")
def generate(question: str, context_docs: list[dict]) -> str:
    context_text = " ".join(d["text"] for d in context_docs)
    return f"Based on company policy: {context_text[:120]}..."

In [ ]:
from opik import opik_context

@opik.track(project_name=OPIK_PROJECT_NAME)
def rag_pipeline(question: str) -> str:
    context_docs = retrieve(question)
    answer = generate(question, context_docs)

    # Supporting context goes in metadata so reviewers can verify the answer without it crowding the output
    opik_context.update_current_trace(
        metadata={
            "retrieved_context": [d["text"] for d in context_docs],
            "sources": [d["source"] for d in context_docs],
        }
    )

    return answer


for question in QUESTIONS:
    rag_pipeline(question)

opik.flush_tracker()
print(f"Logged {len(QUESTIONS)} traces to project '{OPIK_PROJECT_NAME}'")

## 2. Creating an annotation queue

With traces structured correctly, we can create a queue and populate it programmatically. Queues can also be created in the Opik UI — the SDK approach is useful for CI pipelines and batch review workflows.

See the [annotation queues docs](https://www.comet.com/docs/opik/evaluation/advanced/annotation_queues) for queue management options including sharing queues with SMEs.

In [ ]:
client = opik.Opik()

queue = client.create_traces_annotation_queue(
    name="RAG Answer Quality Review",
    instructions=(
        "Score each answer for accuracy and relevance. "
        "Expand the Metadata section to see the retrieved context used to generate the answer."
    ),
    feedback_definition_names=["relevance", "accuracy"],
)

traces = client.search_traces(project_name=OPIK_PROJECT_NAME)
queue.add_traces(traces)

print(f"Queue '{queue.name}' created")
print(f"Added {len(traces)} traces for review")

## 3. Post-hoc enrichment

If traces were logged without context in metadata — or if the context comes from a dataset row rather than the live pipeline — you can enrich traces after the fact using `client.update_trace()`.

> **Why the cell below logs a `may cause data loss` warning:** Opik batches writes, and this demo updates traces that were created seconds earlier in the same run. Updating a trace before its creation has been committed can drop the update. In a real post-hoc workflow the traces come from an earlier session and are already committed, so the warning doesn't apply. The `client.flush()` at the end commits the enrichment writes before the traces are added to a queue. See [batching and updates](https://www.comet.com/docs/opik/tracing/batching_and_updates) for the recommended patterns.

In [ ]:
# Simulate a dataset that has ground-truth context for each question
dataset_context = {
    "What is our leave policy?": "Employees receive 25 days annual leave. Requests need 2 weeks notice.",
    "How do I request flexible working?": "Submit form FW-01 to your line manager. Decisions issued within 28 days.",
    "What are the performance review criteria?": "Four dimensions: delivery, collaboration, growth, and values.",
}

for trace in traces:
    question = trace.input.get("question", "") if isinstance(trace.input, dict) else ""
    context = dataset_context.get(question)
    if context:
        client.update_trace(
            trace_id=trace.id,
            project_name=OPIK_PROJECT_NAME,
            metadata={"dataset_context": context},
        )

client.flush()
print("Metadata enrichment complete")

## Summary

Where data lives in a trace, and why:

| Field | What to log | Why |
|---|---|---|
| `input` | The user's question or request | The reviewer needs to see what was asked |
| `output` | The final answer only | Keeps the review surface focused on what's being scored |
| `metadata` | Retrieval context, sources | Supporting detail a reviewer can check, without crowding the output |
| child `spans` | Every sub-step (`@opik.track`) | Full technical detail, preserved and inspectable |

**Key takeaways:**

1. Log only the final answer in `output` — never the full pipeline state.
2. Put retrieval context in `metadata` using `opik_context.update_current_trace(metadata=...)` — it stays out of the way but remains available to reviewers.
3. Decorate sub-steps with `@opik.track` — they become child spans, preserving full technical detail without crowding the output.
4. For existing traces, enrich with `client.update_trace()` and call `client.flush()` before adding to a queue.

In [ ]:
try:
    queue.delete()
    print("Queue deleted")
except Exception as e:
    print(f"Cleanup error: {e}")